In [ ]:
# Google Drive'i baglar (Colab disinda calisirsa atlar). Model dosyasi buraya kaydedilecek.
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    KAYIT_KLASORU = '/content/drive/MyDrive'
except ImportError:
    KAYIT_KLASORU = '.'

In [ ]:
# Veriyi yukler, boyutunu, eksik ve tekrar eden degerlerini inceler; tekrar edenleri siler ve bmi aykirilarini sadece sayar (silmez).
import pandas as pd
import numpy as np

df = pd.read_csv('/content/insurance.csv')
display(df.head())
display(df.shape)
display(df.info())
display(df.describe())
print("Eksik degerler:\n", df.isnull().sum())
print("Tekrar eden satir sayisi:", df.duplicated().sum())

df = df.drop_duplicates()

q1, q3 = df['bmi'].quantile(0.25), df['bmi'].quantile(0.75)
iqr = q3 - q1
aykiri = (df['bmi'] < q1 - 1.5 * iqr) | (df['bmi'] > q3 + 1.5 * iqr)
print(f"BMI aykiri deger sayisi: {aykiri.sum()} (gercekci degerler oldugu icin silinmedi)")

In [ ]:
# Bagimsiz degiskenleri (X) ve orijinal olcekteki hedef degiskeni (y = charges) tanimlar. Log donusumu pipeline icinde yapilacak.
X = df.drop(columns=['charges'])
y = df['charges']

In [ ]:
# Veriyi train/test olarak ayirir. Bu islemden once veri uzerinde hicbir ogrenme yapilmaz (veri sizintisi onlemi).
# Regresyonda da stratify yapilabilir: smoker ve charges'in quartile gruplarini birlestirip etiket olarak kullaniriz.
from sklearn.model_selection import train_test_split

y_bins = pd.qcut(y, q=4, labels=False).astype(str)
stratify_col = X['smoker'].astype(str) + "_" + y_bins

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=stratify_col
)
print("Train:", X_train.shape, "| Test:", X_test.shape)
print("Smoker orani -> train:", round((X_train['smoker'] == 'yes').mean(), 3), "| test:", round((X_test['smoker'] == 'yes').mean(), 3))

In [ ]:
# Sadece X_train icindeki sayisal sutunlar arasindaki coklu dogrusal baglantiyi (VIF) add_constant ile analiz eder.
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

sayisal_sutunlar = X_train.select_dtypes(include=[np.number]).columns.tolist()
X_vif = sm.add_constant(X_train[sayisal_sutunlar])
vif_data = pd.DataFrame({
    'ozellik': X_vif.columns,
    'VIF': [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
})
display(vif_data[vif_data['ozellik'] != 'const'])

In [ ]:
# On isleme, log hedef donusumu, modeller ve pipeline'lari tanimlar. On isleme modelle ayni pipeline icinde oldugu icin
# her CV fold'unda sadece o fold'un egitim kismina fit edilir (sizinti yok). Skorlar orijinal olcekte hesaplanir.
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.preprocessing import OneHotEncoder, StandardScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.linear_model import ElasticNet
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.model_selection import KFold

preprocessor = ColumnTransformer(transformers=[
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), ['sex', 'smoker', 'region']),
    ('num', StandardScaler(), ['age', 'bmi', 'children'])
])

# Dogrusal model smoker x bmi gibi etkilesimleri kendi ogrenemedigi icin etkilesim terimleri eklenir
linear_preprocessor = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('interaction', PolynomialFeatures(degree=2, interaction_only=True, include_bias=False))
])

def log_hedef(model):
    return TransformedTargetRegressor(regressor=model, func=np.log1p, inverse_func=np.expm1)

pipelines = {
    'ElasticNet': Pipeline([('prep', linear_preprocessor), ('model', log_hedef(ElasticNet(random_state=42, max_iter=10000)))]),
    'RandomForest': Pipeline([('prep', preprocessor), ('model', log_hedef(RandomForestRegressor(random_state=42)))]),
    'XGBoost': Pipeline([('prep', preprocessor), ('model', log_hedef(XGBRegressor(random_state=42)))])
}

cv_strategy = KFold(n_splits=5, shuffle=True, random_state=42)

In [ ]:
# Ayarsiz (baseline) modellerin CV sonuclarini TRAIN ve VALIDATION R2 olarak yan yana gosterir; overfitting buradan da gorulur.
from sklearn.model_selection import cross_validate

baseline = []
for isim, pipe in pipelines.items():
    s = cross_validate(pipe, X_train, y_train, cv=cv_strategy, scoring='r2', return_train_score=True, n_jobs=-1)
    baseline.append({'Model': isim, 'Train_R2': s['train_score'].mean(), 'CV_R2': s['test_score'].mean(),
                     'Gap': s['train_score'].mean() - s['test_score'].mean()})
display(pd.DataFrame(baseline))

In [ ]:
# Uc modelin hiperparametrelerini CV ile optimize eder (ElasticNet: GridSearch, RF ve XGBoost: RandomizedSearch).
# Grid'ler duzenleyici parametreler icerir; en iyi deger aralik kenarina denk gelirse uyari verilir.
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

scoring = {'r2': 'r2', 'neg_mean_absolute_error': 'neg_mean_absolute_error',
           'neg_root_mean_squared_error': 'neg_root_mean_squared_error'}
ortak = dict(cv=cv_strategy, scoring=scoring, refit='neg_root_mean_squared_error', return_train_score=True, n_jobs=-1)

param_gridler = {
    'ElasticNet': {
        'model__regressor__alpha': [0.0001, 0.001, 0.01, 0.1, 1],
        'model__regressor__l1_ratio': [0.1, 0.5, 0.9]
    },
    'RandomForest': {
        'model__regressor__n_estimators': [200, 400],
        'model__regressor__max_depth': [3, 5, 8, None],
        'model__regressor__min_samples_leaf': [1, 3, 5, 10, 20],
        'model__regressor__max_features': ['sqrt', 0.7, 1.0]
    },
    'XGBoost': {
        'model__regressor__n_estimators': [100, 200, 400],
        'model__regressor__learning_rate': [0.03, 0.05, 0.1],
        'model__regressor__max_depth': [2, 3, 4, 5],
        'model__regressor__subsample': [0.7, 0.85, 1.0],
        'model__regressor__reg_lambda': [1, 5, 10],
        'model__regressor__min_child_weight': [1, 5]
    }
}

searches = {}
for isim, pipe in pipelines.items():
    print(f"{isim} araniyor...")
    if isim == 'ElasticNet':
        arama = GridSearchCV(pipe, param_gridler[isim], **ortak)
    else:
        arama = RandomizedSearchCV(pipe, param_gridler[isim], n_iter=30, random_state=42, **ortak)
    searches[isim] = arama.fit(X_train, y_train)

    print("  En iyi parametreler:", {k.split('__')[-1]: v for k, v in arama.best_params_.items()})
    for p, v in arama.best_params_.items():
        secenekler = [s for s in param_gridler[isim][p] if s is not None]
        if v is not None and len(secenekler) > 1 and v in (secenekler[0], secenekler[-1]):
            print(f"  UYARI: {p.split('__')[-1]}={v} aralik kenarinda, grid genisletilebilir.")

In [ ]:
# ADIM 1 - OVERFITTING KONTROLU (metriklerden once): Train, CV ve Test R2 yan yana; gap > 0.10 ise OVERFIT.
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

overfit_satirlari, perf_satirlari = [], []
for isim, arama in searches.items():
    i = arama.best_index_
    res = arama.cv_results_
    train_r2 = res['mean_train_r2'][i]
    cv_r2 = res['mean_test_r2'][i]
    tahmin = arama.predict(X_test)
    test_r2 = r2_score(y_test, tahmin)

    gap_cv, gap_test = train_r2 - cv_r2, train_r2 - test_r2
    durum = 'OVERFIT' if (gap_cv > 0.10 or gap_test > 0.10) else 'NORMAL'
    overfit_satirlari.append({'Model': isim, 'Train_R2': train_r2, 'CV_R2': cv_r2, 'Test_R2': test_r2,
                              'Gap_Train_CV': gap_cv, 'Gap_Train_Test': gap_test, 'Durum': durum})

    perf_satirlari.append({'Model': isim,
                           'CV_R2': cv_r2, 'CV_MAE': -res['mean_test_neg_mean_absolute_error'][i],
                           'CV_RMSE': -res['mean_test_neg_root_mean_squared_error'][i],
                           'Test_R2': test_r2, 'Test_MAE': mean_absolute_error(y_test, tahmin),
                           'Test_RMSE': np.sqrt(mean_squared_error(y_test, tahmin))})

overfit_df = pd.DataFrame(overfit_satirlari)
perf_df = pd.DataFrame(perf_satirlari)
display(overfit_df.round(4))

In [ ]:
# ADIM 2 - PERFORMANS: Uc modelin R2, MAE ve RMSE degerlerini (orijinal olcek) CV ve test icin tek tabloda gosterir.
display(perf_df.round(4))

In [ ]:
# ADIM 3 - OTOMATIK MODEL SECIMI: OVERFIT olmayanlar arasindan en dusuk CV RMSE; fark %2'den kucukse dusuk gap'li, o da esitse basit model.
# Test seti secimde kullanilmaz, sadece CV ile tutarli mi diye kontrol edilir. Kazanan pipeline kaydedilir.
import joblib

basitlik = {'ElasticNet': 0, 'RandomForest': 1, 'XGBoost': 2}
adaylar = overfit_df[overfit_df['Durum'] == 'NORMAL']['Model'].tolist()
if not adaylar:
    print("UYARI: Tum modeller OVERFIT, en dusuk gap'li model seciliyor.")
    adaylar = [overfit_df.sort_values('Gap_Train_CV').iloc[0]['Model']]

p = perf_df[perf_df['Model'].isin(adaylar)].sort_values('CV_RMSE').reset_index(drop=True)
en_iyi_rmse = p.loc[0, 'CV_RMSE']
yakinlar = p[p['CV_RMSE'] <= en_iyi_rmse * 1.02]['Model'].tolist()
gaps = overfit_df.set_index('Model')['Gap_Train_CV']
secilen = sorted(yakinlar, key=lambda m: (round(gaps[m], 3), basitlik[m]))[0]

gerekce = f"CV RMSE'si en dusuk modelin %2'si icindeki adaylar: {yakinlar}. Bunlar arasinda en dusuk train-CV gap'li: {secilen}."
print(f">>> Secilen model: {secilen}")
print("Gerekce:", gerekce)
print(overfit_df[overfit_df['Model'] == secilen][['Train_R2', 'CV_R2', 'Test_R2', 'Gap_Train_CV']].round(4).to_string(index=False))
print(perf_df[perf_df['Model'] == secilen][['CV_RMSE', 'Test_RMSE']].round(1).to_string(index=False))

if perf_df.sort_values('CV_RMSE')['Model'].tolist() != perf_df.sort_values('Test_RMSE')['Model'].tolist():
    print("\nUYARI: Test siralamasi CV siralamasindan farkli. Model teste gore yeniden secilmedi (test sizintisi olmamasi icin).")

joblib.dump(searches[secilen].best_estimator_, f'{KAYIT_KLASORU}/best_model.joblib')
print(f"Model kaydedildi: {KAYIT_KLASORU}/best_model.joblib")